<a href="https://colab.research.google.com/github/maruwagner893-droid/cloud-provider-analytics/blob/main/notebooks/02_pipeline_etapa2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cloud-provider-analytics")
    .master("local[*]")
    .getOrCreate()
)
print("Versión de Spark:", spark.version)

Versión de Spark: 4.0.5


In [3]:
import os

REPO = "/content/cloud-provider-analytics"

if not os.path.exists(REPO):
    !git clone https://github.com/maruwagner893-droid/cloud-provider-analytics {REPO}

!unzip -q -o {REPO}/data/cloud_provider_challenge_dataset_v1.zip -d {REPO}/data/extracted

# Configuración del pipeline: rutas y parámetros en config/pipeline.yml
import yaml
config = yaml.safe_load(open(os.path.join(REPO, "config/pipeline.yml"), encoding="utf-8"))
RUTAS = config["rutas"]

def ruta(nombre):
    return os.path.join(REPO, RUTAS[nombre])

spark.conf.set("spark.sql.shuffle.partitions", config["spark"]["shuffle_partitions"])
spark.conf.set("spark.sql.session.timeZone", config["spark"]["timezone"])
print(config)

Cloning into '/content/cloud-provider-analytics'...
remote: Enumerating objects: 1929, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 1929 (delta 89), reused 43 (delta 43), pack-reused 1795 (from 1)
Receiving objects: 100% (1929/1929), 6.73 MiB | 27.92 MiB/s, done.
Resolving deltas: 100% (1128/1128), done.
{'spark': {'shuffle_partitions': 8, 'timezone': 'UTC'}, 'rutas': {'landing': 'data/extracted/datalake/landing', 'bronze': 'data/bronze', 'bronze_eventos': 'data/bronze/usage_events_stream', 'silver_eventos': 'data/silver/usage_events', 'silver_billing': 'data/silver/billing_monthly', 'quarantine_eventos': 'data/quarantine/usage_events', 'quarantine_billing': 'data/quarantine/billing_monthly', 'gold_uso': 'data/gold/org_daily_usage_by_service', 'gold_anomalias': 'data/gold/cost_anomaly_mart', 'gold_servicios_14d': 'data/gold/org_service_cost_14d', 'gold_revenue': 'data/gold/revenue_by_org_month', 'gold_genai': 'data

In [4]:
LANDING = ruta("landing")
print("Landing:", LANDING)
print(sorted(os.listdir(LANDING)))

Landing: /content/cloud-provider-analytics/data/extracted/datalake/landing
['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'usage_events_stream', 'users.csv']


In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, DateType, DoubleType
)

schema_customers = StructType([
    StructField("org_id",           StringType(),  True),
    StructField("org_name",         StringType(),  True),
    StructField("industry",         StringType(),  True),
    StructField("hq_region",        StringType(),  True),
    StructField("plan_tier",        StringType(),  True),
    StructField("is_enterprise",    BooleanType(), True),
    StructField("signup_date",      DateType(),    True),
    StructField("sales_rep",        StringType(),  True),
    StructField("lifecycle_stage",  StringType(),  True),
    StructField("marketing_source", StringType(),  True),
    StructField("nps_score",        DoubleType(),  True),
])

customers = (
    spark.read
    .option("header", True)
    .schema(schema_customers)
    .csv(os.path.join(LANDING, "customers_orgs.csv"))
)

customers.printSchema()
customers.show(5, truncate=False)

# Verificación
print("Filas:", customers.count())
print("signup_date nulos:", customers.filter(F.col("signup_date").isNull()).count())
print("nps_score nulos:", customers.filter(F.col("nps_score").isNull()).count())

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)

+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+
|org_xaji0y6d|Nimbus Labs 0  |Education    |sa-east   |standard |true         |2025-05-26 |rep_

In [6]:
BRONZE = ruta("bronze")

customers_bronze = (
    customers
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
)

(
    customers_bronze
    .coalesce(1)                      # 80 filas: un solo archivo, sin archivos chicos
    .write.mode("overwrite")          # re-ejecutable: pisa el resultado anterior
    .partitionBy("plan_tier")
    .parquet(os.path.join(BRONZE, "customers_orgs"))
)

# Verificación: leer lo que se guardó
check = spark.read.parquet(os.path.join(BRONZE, "customers_orgs"))
check.printSchema()
check.select("org_id", "ingest_ts", "source_file").show(3, truncate=False)
print("Filas en Bronze:", check.count())

!ls -la {BRONZE}/customers_orgs

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- plan_tier: string (nullable = true)

+------------+--------------------------+------------------+
|org_id      |ingest_ts                 |source_file       |
+------------+--------------------------+------------------+
|org_xaji0y6d|2026-11-24 18:36:30.013402|customers_orgs.csv|
|org_pbhsahxt|2026-11-24 18:36:30.013402|customers_orgs.csv|
|org_8mdd4v30|2026-11-24 18:36:30.013402|customers_orgs.csv|
+------------+--------------------------+------------------+
only showing top

In [7]:
schema_users = StructType([
    StructField("user_id",    StringType(),  True),
    StructField("org_id",     StringType(),  True),
    StructField("email",      StringType(),  True),
    StructField("role",       StringType(),  True),
    StructField("active",     BooleanType(), True),
    StructField("created_at", DateType(),    True),
    StructField("last_login", DateType(),    True),
])

schema_billing = StructType([
    StructField("invoice_id",           StringType(), True),
    StructField("org_id",               StringType(), True),
    StructField("month",                DateType(),   True),
    StructField("subtotal",             DoubleType(), True),
    StructField("credits",              DoubleType(), True),
    StructField("taxes",                DoubleType(), True),
    StructField("currency",             StringType(), True),
    StructField("exchange_rate_to_usd", DoubleType(), True),
])

schema_tickets = StructType([
    StructField("ticket_id",    StringType(),  True),
    StructField("org_id",       StringType(),  True),
    StructField("category",     StringType(),  True),
    StructField("severity",     StringType(),  True),
    StructField("created_at",   DateType(),    True),
    StructField("resolved_at",  DateType(),    True),
    StructField("csat",         DoubleType(),  True),
    StructField("sla_breached", BooleanType(), True),
])

def ingest_master(name, schema, key, part_col):
    """Landing (CSV) -> Bronze (Parquet): tipifica, deduplica por clave y agrega metadatos."""
    df = (
        spark.read.option("header", True).schema(schema)
        .csv(os.path.join(LANDING, f"{name}.csv"))
        .withColumn("ingest_ts", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_name"))
    )
    n_in = df.count()
    df = df.dropDuplicates([key])
    n_out = df.count()
    out = os.path.join(BRONZE, name)
    df.coalesce(1).write.mode("overwrite").partitionBy(part_col).parquet(out)
    print(f"{name}: leídas={n_in}  duplicadas quitadas={n_in - n_out}  guardadas={n_out}  -> {out}")

ingest_master("customers_orgs",  schema_customers, "org_id",     "plan_tier")
ingest_master("users",           schema_users,     "user_id",    "role")
ingest_master("billing_monthly", schema_billing,   "invoice_id", "month")
ingest_master("support_tickets", schema_tickets,   "ticket_id",  "severity")
# Verificación
users_b   = spark.read.parquet(os.path.join(BRONZE, "users"))
billing_b = spark.read.parquet(os.path.join(BRONZE, "billing_monthly"))
print("users last_login nulos:", users_b.filter(F.col("last_login").isNull()).count())
print("billing credits nulos:", billing_b.filter(F.col("credits").isNull()).count())
print("billing subtotal < 0:", billing_b.filter(F.col("subtotal") < 0).count())

customers_orgs: leídas=80  duplicadas quitadas=0  guardadas=80  -> /content/cloud-provider-analytics/data/bronze/customers_orgs
users: leídas=800  duplicadas quitadas=0  guardadas=800  -> /content/cloud-provider-analytics/data/bronze/users
billing_monthly: leídas=240  duplicadas quitadas=0  guardadas=240  -> /content/cloud-provider-analytics/data/bronze/billing_monthly
support_tickets: leídas=1000  duplicadas quitadas=0  guardadas=1000  -> /content/cloud-provider-analytics/data/bronze/support_tickets
users last_login nulos: 139
billing credits nulos: 137
billing subtotal < 0: 13


In [8]:
for m in ["customers_orgs", "users", "billing_monthly", "support_tickets"]:
    print("==", m)
    for carpeta, _, archivos in sorted(os.walk(os.path.join(BRONZE, m))):
        for a in sorted(archivos):
            if a.endswith(".parquet"):
                p = os.path.join(carpeta, a)
                print(os.path.relpath(p, BRONZE), os.path.getsize(p), "bytes")

== customers_orgs
customers_orgs/plan_tier=enterprise/part-00000-be44fe86-bcd7-400a-9d79-3f3ba946111a.c000.snappy.parquet 4437 bytes
customers_orgs/plan_tier=free/part-00000-be44fe86-bcd7-400a-9d79-3f3ba946111a.c000.snappy.parquet 4354 bytes
customers_orgs/plan_tier=pro/part-00000-be44fe86-bcd7-400a-9d79-3f3ba946111a.c000.snappy.parquet 4868 bytes
customers_orgs/plan_tier=standard/part-00000-be44fe86-bcd7-400a-9d79-3f3ba946111a.c000.snappy.parquet 5442 bytes
== users
users/role=admin/part-00000-2e20b5d4-5d0c-4a53-852d-28bbd43508e1.c000.snappy.parquet 5847 bytes
users/role=analyst/part-00000-2e20b5d4-5d0c-4a53-852d-28bbd43508e1.c000.snappy.parquet 6609 bytes
users/role=data_engineer/part-00000-2e20b5d4-5d0c-4a53-852d-28bbd43508e1.c000.snappy.parquet 9007 bytes
users/role=developer/part-00000-2e20b5d4-5d0c-4a53-852d-28bbd43508e1.c000.snappy.parquet 7073 bytes
users/role=devops/part-00000-2e20b5d4-5d0c-4a53-852d-28bbd43508e1.c000.snappy.parquet 7935 bytes
users/role=ml_engineer/part-00000

In [9]:
from pyspark.sql.types import TimestampType, IntegerType, LongType


schema_events = StructType([
    StructField("event_id",           StringType(),    True),
    StructField("timestamp",          TimestampType(), True),
    StructField("org_id",             StringType(),    True),
    StructField("resource_id",        StringType(),    True),
    StructField("service",            StringType(),    True),
    StructField("region",             StringType(),    True),
    StructField("metric",             StringType(),    True),
    StructField("value",              StringType(),    True),   # llega como número o como texto
    StructField("unit",               StringType(),    True),
    StructField("cost_usd_increment", DoubleType(),    True),
    StructField("schema_version",     IntegerType(),   True),
    StructField("carbon_kg",          DoubleType(),    True),   # solo en v2
    StructField("genai_tokens",       LongType(),      True),   # solo servicio genai
])

EVENTS_LANDING = os.path.join(LANDING, "usage_events_stream")

# Cada archivo original trae eventos de todo el período, no de un tramo
originales = (
    spark.read.schema(schema_events).json(EVENTS_LANDING)
    .withColumn("f", F.col("_metadata.file_name"))
)
rangos = (
    originales.groupBy("f")
    .agg(F.count("*").alias("n"), F.min("timestamp").alias("desde"), F.max("timestamp").alias("hasta"))
    .orderBy("f")
)
rangos.show(5, truncate=False)
print("Archivos:", rangos.count())
print("Eventos totales:", originales.count())
print("event_id distintos:", originales.select("event_id").distinct().count())

+----------------------+---+-------------------+-------------------+
|f                     |n  |desde              |hasta              |
+----------------------+---+-------------------+-------------------+
|events_part_0000.jsonl|360|2025-07-03 00:08:00|2025-08-31 16:32:00|
|events_part_0001.jsonl|360|2025-07-03 02:38:00|2025-08-31 23:51:00|
|events_part_0002.jsonl|360|2025-07-03 14:16:00|2025-08-31 20:47:00|
|events_part_0003.jsonl|360|2025-07-03 00:48:00|2025-08-31 23:51:00|
|events_part_0004.jsonl|360|2025-07-03 10:43:00|2025-08-31 20:40:00|
+----------------------+---+-------------------+-------------------+
only showing top 5 rows
Archivos: 120
Eventos totales: 43200
event_id distintos: 43200


In [10]:
BRONZE_EVENTS = ruta("bronze_eventos")
CHECKPOINT    = os.path.join(ruta("checkpoints"), "bronze_events")
stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", config["streaming"]["max_files_per_trigger"])
    .json(EVENTS_LANDING)                     # archivos originales, en el orden en que llegan
)

bronze_stream = (
    stream
    .withColumn("value", F.col("value").cast("double"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withWatermark("ingest_ts", config["streaming"]["watermark"])
    .dropDuplicatesWithinWatermark(["event_id"])
)

def run_bronze_events():
    q = (
        bronze_stream.writeStream
        .queryName("bronze_events")
        .format("parquet")
        .outputMode("append")
        .option("path", BRONZE_EVENTS)
        .option("checkpointLocation", CHECKPOINT)
        .partitionBy("year", "month")
        .trigger(availableNow=True)                             # procesa todo lo pendiente y termina
        .start()
    )
    q.awaitTermination()
    return q

def contar_bronze():
    d = spark.read.parquet(BRONZE_EVENTS)
    print("Eventos en Bronze:", d.count())
    d.groupBy("year", "month").count().orderBy("year", "month").show()

# Monitoreo de cada micro-lote con StreamingQueryListener (Clase 03)
from pyspark.sql.streaming import StreamingQueryListener
import time

lotes = []   # métricas de cada micro-lote

class Monitor(StreamingQueryListener):
    def onQueryStarted(self, event):
        print("inicio", event.id, event.name)
    def onQueryProgress(self, event):
        p = event.progress
        lotes.append((p.batchId, p.numInputRows, p.inputRowsPerSecond, p.processedRowsPerSecond))
    def onQueryIdle(self, event):
        pass
    def onQueryTerminated(self, event):
        print("fin", event.id, event.exception)

spark.streams.addListener(Monitor())

# Primera ejecución
consulta = run_bronze_events()
time.sleep(2)                          # los eventos del listener llegan con un pequeño retraso

for batch_id, filas, entrada, procesadas in lotes:
    print(batch_id, filas, entrada, procesadas)
print("Micro-lotes con datos:", sum(1 for l in lotes if l[1] > 0))
print("Filas leídas en total:", sum(l[1] for l in lotes))

contar_bronze()

inicio 4d075beb-b60e-4b96-b5b3-6ffc35d3928d bronze_events
fin 4d075beb-b60e-4b96-b5b3-6ffc35d3928d None
0 3600 0.0 707.9646017699115
1 3600 698.6221618474675 1674.418604651163
2 3600 1669.7588126159553 1568.6274509803923
3 3600 1566.579634464752 1590.808661069377
4 3600 1586.6020273248125 1858.544140423335
5 3600 1851.851851851852 1237.538673083534
6 3600 1233.721727210418 1303.4033309196234
7 3600 1301.5184381778743 1987.8520154610712
8 3600 1978.021978021978 2045.4545454545455
9 3600 2040.8163265306123 1920.0
10 3600 1916.932907348243 2139.03743315508
11 3600 2133.965619442798 2086.9565217391305
12 0 0.0 0.0
Micro-lotes con datos: 12
Filas leídas en total: 43200
Eventos en Bronze: 43200
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    7|20880|
|2025|    8|22320|
+----+-----+-----+



In [11]:
import shutil

def total_bronze():
    return spark.read.parquet(BRONZE_EVENTS).count()

# A) Re-ejecutar sin archivos nuevos: el checkpoint evita reprocesar
antes = total_bronze()
run_bronze_events()
despues = total_bronze()
print(f"A) Re-ejecución sin archivos nuevos -> antes={antes}  después={despues}")

# B) Llega un archivo repetido (mismos event_id): no debe sumar eventos
ultimo = os.path.join(EVENTS_LANDING, "events_part_0119.jsonl")
repetido = os.path.join(EVENTS_LANDING, "events_part_9999_repetido.jsonl")
shutil.copy(ultimo, repetido)

antes = total_bronze()
run_bronze_events()
despues = total_bronze()
print(f"B) Llega un archivo repetido      -> antes={antes}  después={despues}")

os.remove(repetido)   # Landing queda como estaba

print("Checkpoint:", os.listdir(CHECKPOINT))

inicio 4d075beb-b60e-4b96-b5b3-6ffc35d3928d bronze_events
fin 4d075beb-b60e-4b96-b5b3-6ffc35d3928d None
A) Re-ejecución sin archivos nuevos -> antes=43200  después=43200
inicio 4d075beb-b60e-4b96-b5b3-6ffc35d3928d bronze_events
fin 4d075beb-b60e-4b96-b5b3-6ffc35d3928d None
B) Llega un archivo repetido      -> antes=43200  después=43200
Checkpoint: ['.metadata.crc', 'state', 'commits', 'offsets', 'sources', 'metadata']


In [12]:
SILVER_EVENTS     = ruta("silver_eventos")
QUARANTINE_EVENTS = ruta("quarantine_eventos")
CHECKPOINT_SILVER = os.path.join(ruta("checkpoints"), "silver_events")
COSTO_MINIMO      = config["calidad"]["costo_minimo"]

customers_dim = (
    spark.read.parquet(os.path.join(BRONZE, "customers_orgs"))
    .select("org_id", "industry", F.col("hq_region").alias("org_hq_region"),
            "plan_tier", "is_enterprise")
)

unidad_por_metrica = (
    F.when(F.col("metric") == "cpu_hours", "hours")
     .when(F.col("metric") == "requests", "count")
     .when(F.col("metric") == "storage_gb_hours", "gb_hours")
)

def procesar_lote(batch_df, batch_id):
    motivo = (
        F.when(F.col("event_id").isNull(), "event_id_nulo")
        .when(F.col("cost_usd_increment") < COSTO_MINIMO, f"cost_usd_increment_menor_a_{COSTO_MINIMO}")
    )
    marcado = batch_df.withColumn("quarantine_reason", motivo)

    invalidos = (
        marcado.filter(F.col("quarantine_reason").isNotNull())
        .withColumn("quarantine_ts", F.current_timestamp())
    )
    validos = marcado.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")

    silver = (
        validos
        .withColumn("unit_imputed", F.col("unit").isNull())
        .withColumn("unit", F.coalesce(F.col("unit"), unidad_por_metrica))
        .withColumn("value_missing", F.col("value").isNull())
        .withColumn("usage_date", F.to_date("timestamp"))
        .withColumn("requests",         F.when(F.col("metric") == "requests",         F.col("value")))
        .withColumn("cpu_hours",        F.when(F.col("metric") == "cpu_hours",        F.col("value")))
        .withColumn("storage_gb_hours", F.when(F.col("metric") == "storage_gb_hours", F.col("value")))
        .join(F.broadcast(customers_dim), on="org_id", how="left")
    )

    silver.coalesce(1).write.mode("append").partitionBy("year", "month").parquet(SILVER_EVENTS)
    if invalidos.count() > 0:                                   # solo escribe si hay inválidos
        invalidos.coalesce(1).write.mode("append").parquet(QUARANTINE_EVENTS)
bronze_schema = spark.read.parquet(BRONZE_EVENTS).schema
silver_stream = (
    spark.readStream.schema(bronze_schema)
    .option("maxFilesPerTrigger", config["streaming"]["max_files_per_trigger"])
    .parquet(BRONZE_EVENTS)
)

def run_silver_events():
    q = (
        silver_stream.writeStream
        .foreachBatch(procesar_lote)
        .option("checkpointLocation", CHECKPOINT_SILVER)
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()

run_silver_events()

# Verificación
s = spark.read.parquet(SILVER_EVENTS)
q = spark.read.parquet(QUARANTINE_EVENTS)
n_s, n_q = s.count(), q.count()
print("Silver:", n_s)
print("Quarantine:", n_q)
print("% inválidos:", round(n_q / (n_s + n_q) * 100, 2))
s.groupBy("year", "month").count().orderBy("year", "month").show()
print("unit nulos en Silver:", s.filter(F.col("unit").isNull()).count())
print("unit imputada:", s.filter("unit_imputed").count())
print("value nulo:", s.filter("value_missing").count())
q.groupBy("quarantine_reason").count().show(truncate=False)
q.select("event_id", "cost_usd_increment", "quarantine_reason", "source_file") \
 .orderBy("cost_usd_increment").show(5, truncate=False)
s.select("event_id", "service", "metric", "unit", "requests", "cpu_hours",
         "storage_gb_hours", "industry", "plan_tier").show(5, truncate=False)

inicio 33522ed2-5356-4b13-8e5b-ecd7f0260d51 None
fin 33522ed2-5356-4b13-8e5b-ecd7f0260d51 None
Silver: 42989
Quarantine: 211
% inválidos: 0.49
+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2025|    7|20778|
|2025|    8|22211|
+----+-----+-----+

unit nulos en Silver: 0
unit imputada: 2069
value nulo: 873
+--------------------------------+-----+
|quarantine_reason               |count|
+--------------------------------+-----+
|cost_usd_increment_menor_a_-0.01|211  |
+--------------------------------+-----+

+----------------+------------------+--------------------------------+----------------------+
|event_id        |cost_usd_increment|quarantine_reason               |source_file           |
+----------------+------------------+--------------------------------+----------------------+
|evt_t5brta67j5l3|-154.4608         |cost_usd_increment_menor_a_-0.01|events_part_0059.jsonl|
|evt_9ur456xoexl5|-19.8245          |cost_usd_increment_menor_a_-0.01|events_part_0104.jsonl|
|evt_z

In [13]:
# Silver de billing_monthly: créditos, moneda normalizada a USD y regla de calidad
SILVER_BILLING     = ruta("silver_billing")
QUARANTINE_BILLING = ruta("quarantine_billing")

billing_bronze = spark.read.parquet(os.path.join(BRONZE, "billing_monthly"))

tasa_valida = F.coalesce(F.col("exchange_rate_to_usd") > 0, F.lit(False))

billing_marcado = (
    billing_bronze
    .withColumn("credits_imputed", F.col("credits").isNull())           # marca cuáles se completaron
    .withColumn("credits", F.coalesce(F.col("credits"), F.lit(0.0)))
    .withColumn("is_credit", F.col("subtotal") < 0)                     # las 13 facturas negativas
    .withColumn("net_usd", F.round(
        (F.col("subtotal") - F.col("credits")) * F.col("exchange_rate_to_usd"), 2))
    .withColumn("quarantine_reason",
        F.when(~tasa_valida, "exchange_rate_nula_o_no_positiva"))
)

billing_validos   = billing_marcado.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")
billing_invalidos = (billing_marcado.filter(F.col("quarantine_reason").isNotNull())
                     .withColumn("quarantine_ts", F.current_timestamp()))

# overwrite: re-ejecutable, pisa el resultado anterior
billing_validos.coalesce(1).write.mode("overwrite").partitionBy("month").parquet(SILVER_BILLING)
billing_invalidos.coalesce(1).write.mode("overwrite").parquet(QUARANTINE_BILLING)

# Verificación
s = spark.read.parquet(SILVER_BILLING)
print("Filas en Silver:", s.count())
print("Quarantine:", billing_invalidos.count())
print("credits imputados a 0:", s.filter("credits_imputed").count())
print("Facturas con crédito (subtotal < 0):", s.filter("is_credit").count())
s.groupBy("currency").count().orderBy("currency").show()
total_org = s.filter(F.col("org_id") == "org_xaji0y6d").agg(F.sum("net_usd")).first()[0]
print("Total org_xaji0y6d (USD):", round(total_org, 2))

Filas en Silver: 240
Quarantine: 0
credits imputados a 0: 137
Facturas con crédito (subtotal < 0): 13
+--------+-----+
|currency|count|
+--------+-----+
|     ARS|   51|
|     EUR|   29|
|     USD|  160|
+--------+-----+

Total org_xaji0y6d (USD): 1729.18


In [14]:
SILVER_TICKETS = ruta("silver_tickets")

tickets_bronze = spark.read.parquet(os.path.join(BRONZE, "support_tickets"))

tickets_silver = (
    tickets_bronze
    .withColumn("is_open", F.col("resolved_at").isNull())
)

tickets_silver.write.mode("overwrite").partitionBy("severity").parquet(SILVER_TICKETS)

t = spark.read.parquet(SILVER_TICKETS)
print("Filas en Silver:", t.count())
print("Tickets abiertos (sin resolved_at):", t.filter("is_open").count())
print("csat nulos:", t.filter(F.col("csat").isNull()).count())
print("Tickets con SLA incumplido:", t.filter("sla_breached").count())
t.groupBy("severity").count().orderBy("severity").show()

Filas en Silver: 1000
Tickets abiertos (sin resolved_at): 240
csat nulos: 254
Tickets con SLA incumplido: 95
+--------+-----+
|severity|count|
+--------+-----+
|critical|   56|
|    high|  204|
|     low|  412|
|  medium|  328|
+--------+-----+



In [15]:
GOLD_USAGE = ruta("gold_uso")

# calcular_gold está en src/gold.py, así la usan el notebook y las pruebas de tests/
import sys
sys.path.append(REPO)
from src.gold import calcular_gold

org_daily_usage_by_service = calcular_gold(spark.read.parquet(SILVER_EVENTS))

# Evidencia del shuffle (captura para evidence/)
org_daily_usage_by_service.explain(mode="formatted")

(
    org_daily_usage_by_service
    .write.mode("overwrite")
    .partitionBy("year", "month")            # una carpeta por mes, como Bronze y Silver
    .parquet(GOLD_USAGE)
)

# Verificación
gold = spark.read.parquet(GOLD_USAGE)
print("Filas Gold:", gold.count())
claves = gold.select("org_id", "usage_date", "service").distinct().count()
print("Claves únicas (grano):", claves)
print("Suma n_events:", gold.agg(F.sum("n_events")).first()[0])
print("Costo total:", round(gold.agg(F.sum("daily_cost_usd")).first()[0], 2))
print("Requests total:", gold.agg(F.sum("requests")).first()[0])
print("genai_tokens total:", gold.agg(F.sum("genai_tokens")).first()[0])
print("Filas con genai_tokens:", gold.filter(F.col("genai_tokens").isNotNull()).count())
gold.groupBy("year", "month").count().orderBy("year", "month").show()
gold.orderBy("org_id", "usage_date", "service").show(5, truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (11)
+- HashAggregate (10)
   +- Exchange (9)
      +- HashAggregate (8)
         +- SortAggregate (7)
            +- Sort (6)
               +- Exchange (5)
                  +- SortAggregate (4)
                     +- Sort (3)
                        +- Project (2)
                           +- Scan parquet  (1)


(1) Scan parquet 
Output [12]: [org_id#4137, event_id#4138, service#4141, cost_usd_increment#4146, carbon_kg#4148, genai_tokens#4149L, usage_date#4154, requests#4155, cpu_hours#4156, storage_gb_hours#4157, year#4162, month#4163]
Batched: true
Location: InMemoryFileIndex [file:/content/cloud-provider-analytics/data/silver/usage_events]
ReadSchema: struct<org_id:string,event_id:string,service:string,cost_usd_increment:double,carbon_kg:double,genai_tokens:bigint,usage_date:date,requests:double,cpu_hours:double,storage_gb_hours:double>

(2) Project
Output [10]: [org_id#4137, event_id#4138, service#4141, cost_usd_increment#4146, carbon_kg#4

In [16]:
# Analítica: detección de anomalías de costo con el método IQR (visto en Minería de Datos I)

GOLD_ANOMALIAS = ruta("gold_anomalias")

gold = spark.read.parquet(GOLD_USAGE)

cuartiles = (
    gold.groupBy("org_id", "service")
    .agg(
        F.percentile("daily_cost_usd", 0.25).alias("q1"),
        F.percentile("daily_cost_usd", 0.75).alias("q3"),
        F.count("*").alias("n_dias"),
    )
    .withColumn("iqr", F.col("q3") - F.col("q1"))
    .withColumn("sup", F.col("q3") + 1.5 * F.col("iqr"))
)

cost_anomaly_mart = (
    gold.select("org_id", "usage_date", "service", "daily_cost_usd", "year", "month")
    .join(F.broadcast(cuartiles), on=["org_id", "service"], how="left")
    .withColumn("is_anomaly", F.col("daily_cost_usd") > F.col("sup"))
    .withColumn("q1",  F.round("q1", 4))
    .withColumn("q3",  F.round("q3", 4))
    .withColumn("iqr", F.round("iqr", 4))
    .withColumn("sup", F.round("sup", 4))
)

cost_anomaly_mart.write.mode("overwrite").partitionBy("year", "month").parquet(GOLD_ANOMALIAS)

# Verificación
a = spark.read.parquet(GOLD_ANOMALIAS)
print("Filas:", a.count())
print("Combinaciones org/servicio:", cuartiles.count())
print("Anomalías (costo > q3 + 1.5*iqr):", a.filter("is_anomaly").count())
a.filter("is_anomaly").orderBy(F.col("daily_cost_usd").desc()) \
 .select("org_id", "usage_date", "service", "daily_cost_usd", "q3", "sup") \
 .show(5, truncate=False)

Filas: 11050
Combinaciones org/servicio: 262
Anomalías (costo > q3 + 1.5*iqr): 196
+------------+----------+-------+--------------+-------+-------+
|org_id      |usage_date|service|daily_cost_usd|q3     |sup    |
+------------+----------+-------+--------------+-------+-------+
|org_rixa11dp|2025-08-24|genai  |340.5259      |34.473 |59.5995|
|org_c11ertj5|2025-07-11|genai  |334.6767      |30.7304|57.1145|
|org_tvhhpbmy|2025-07-04|genai  |313.4168      |30.2732|52.9081|
|org_x7eedtjv|2025-07-09|compute|311.255       |21.1819|37.0582|
|org_5935a0l7|2025-07-09|compute|271.6404      |26.8324|49.3971|
+------------+----------+-------+--------------+-------+-------+
only showing top 5 rows


In [17]:
from datetime import timedelta

GOLD_SERVICIOS_14D = ruta("gold_servicios_14d")

gold = spark.read.parquet(GOLD_USAGE)
fin = gold.agg(F.max("usage_date")).first()[0]
inicio = fin - timedelta(days=13)

org_service_cost_14d = (
    gold.filter((F.col("usage_date") >= inicio) & (F.col("usage_date") <= fin))
    .groupBy("org_id", "service")
    .agg(F.round(F.sum("daily_cost_usd"), 4).alias("cost_usd_14d"))
    .withColumn("window_end", F.lit(fin))
)

org_service_cost_14d.write.mode("overwrite").parquet(GOLD_SERVICIOS_14D)

s14 = spark.read.parquet(GOLD_SERVICIOS_14D)
print("Ventana:", inicio, "a", fin)
print("Filas:", s14.count())
s14.filter(F.col("org_id") == "org_dbdw2pcn").orderBy(F.col("cost_usd_14d").desc()).show(truncate=False)

Ventana: 2025-08-18 a 2025-08-31
Filas: 262
+------------+----------+------------+----------+
|org_id      |service   |cost_usd_14d|window_end|
+------------+----------+------------+----------+
|org_dbdw2pcn|compute   |278.1017    |2025-08-31|
|org_dbdw2pcn|database  |133.2911    |2025-08-31|
|org_dbdw2pcn|networking|8.395       |2025-08-31|
+------------+----------+------------+----------+



In [18]:
GOLD_REVENUE = ruta("gold_revenue")

billing = spark.read.parquet(SILVER_BILLING)

revenue_by_org_month = (
    billing
    .groupBy("org_id", "month", "currency")
    .agg(
        F.round(F.sum(F.col("subtotal") * F.col("exchange_rate_to_usd")), 2).alias("subtotal_usd"),
        F.round(F.sum(F.col("credits") * F.col("exchange_rate_to_usd")), 2).alias("credits_usd"),
        F.round(F.sum(F.col("taxes") * F.col("exchange_rate_to_usd")), 2).alias("taxes_usd"),
        F.round(F.sum((F.col("subtotal") - F.col("credits") + F.col("taxes")) * F.col("exchange_rate_to_usd")), 2).alias("revenue_usd"),
        F.round(F.avg("exchange_rate_to_usd"), 5).alias("exchange_rate_to_usd"),
        F.count("*").alias("n_facturas"),
    )
)

revenue_by_org_month.write.mode("overwrite").parquet(GOLD_REVENUE)

r = spark.read.parquet(GOLD_REVENUE)
print("Filas:", r.count())
print("Revenue total (USD):", round(r.agg(F.sum("revenue_usd")).first()[0], 2))
r.filter(F.col("org_id") == "org_xaji0y6d").orderBy("month").show(truncate=False)

Filas: 240
Revenue total (USD): 164184.89
+------------+----------+--------+------------+-----------+---------+-----------+--------------------+----------+
|org_id      |month     |currency|subtotal_usd|credits_usd|taxes_usd|revenue_usd|exchange_rate_to_usd|n_facturas|
+------------+----------+--------+------------+-----------+---------+-----------+--------------------+----------+
|org_xaji0y6d|2025-06-01|USD     |1107.09     |0.0        |232.49   |1339.58    |0.97684             |1         |
|org_xaji0y6d|2025-07-01|USD     |621.22      |0.0        |130.46   |751.68     |1.00331             |1         |
|org_xaji0y6d|2025-08-01|ARS     |0.87        |0.0        |0.18     |1.05       |0.0016              |1         |
+------------+----------+--------+------------+-----------+---------+-----------+--------------------+----------+



In [19]:
GOLD_GENAI = ruta("gold_genai")

gold = spark.read.parquet(GOLD_USAGE)

genai_tokens_by_org_date = (
    gold
    .filter((F.col("service") == "genai") & F.col("genai_tokens").isNotNull())
    .select(
        "org_id",
        "usage_date",
        "genai_tokens",
        F.col("daily_cost_usd").alias("cost_usd"),
        "n_events",
    )
)

genai_tokens_by_org_date.write.mode("overwrite").parquet(GOLD_GENAI)

g = spark.read.parquet(GOLD_GENAI)
print("Filas:", g.count())
print("Organizaciones con GenAI:", g.select("org_id").distinct().count())
print("Tokens totales:", g.agg(F.sum("genai_tokens")).first()[0])
g.filter(F.col("org_id") == "org_pbhsahxt").orderBy(F.col("usage_date").desc()).show(5, truncate=False)

Filas: 848
Organizaciones con GenAI: 27
Tokens totales: 2545481
+------------+----------+------------+--------+--------+
|org_id      |usage_date|genai_tokens|cost_usd|n_events|
+------------+----------+------------+--------+--------+
|org_pbhsahxt|2025-08-31|2488        |24.7227 |3       |
|org_pbhsahxt|2025-08-30|5401        |34.0184 |6       |
|org_pbhsahxt|2025-08-29|2210        |16.7402 |3       |
|org_pbhsahxt|2025-08-28|4885        |43.9459 |6       |
|org_pbhsahxt|2025-08-27|1777        |2.5819  |3       |
+------------+----------+------------+--------+--------+
only showing top 5 rows


In [20]:
GOLD_TICKETS = ruta("gold_tickets")

tickets = spark.read.parquet(SILVER_TICKETS)

tickets_by_org_date = (
    tickets
    .groupBy("org_id", F.col("created_at").alias("ticket_date"), "severity")
    .agg(
        F.count("*").alias("n_tickets"),
        F.sum(F.col("sla_breached").cast("int")).alias("n_sla_breached"),
        F.round(F.avg(F.col("sla_breached").cast("double")), 4).alias("sla_breach_rate"),
        F.round(F.avg("csat"), 2).alias("csat_avg"),
        F.sum(F.col("is_open").cast("int")).alias("n_open"),
    )
)

tickets_by_org_date.write.mode("overwrite").parquet(GOLD_TICKETS)

tk = spark.read.parquet(GOLD_TICKETS)
print("Filas:", tk.count())
print("Suma n_tickets:", tk.agg(F.sum("n_tickets")).first()[0])
tk.filter((F.col("org_id") == "org_w3zp08j3") & (F.col("severity") == "critical")).orderBy("ticket_date").show(truncate=False)

Filas: 984
Suma n_tickets: 1000
+------------+-----------+--------+---------+--------------+---------------+--------+------+
|org_id      |ticket_date|severity|n_tickets|n_sla_breached|sla_breach_rate|csat_avg|n_open|
+------------+-----------+--------+---------+--------------+---------------+--------+------+
|org_w3zp08j3|2025-08-08 |critical|1        |0             |0.0            |3.0     |1     |
|org_w3zp08j3|2025-08-23 |critical|1        |0             |0.0            |4.0     |0     |
+------------+-----------+--------+---------+--------------+---------------+--------+------+



In [21]:
def tam(ruta):
    total, n = 0, 0
    for carpeta, _, archivos in os.walk(ruta):
        for a in archivos:
            if a.endswith((".parquet", ".json", ".jsonl")):
                total += os.path.getsize(os.path.join(carpeta, a))
                n += 1
    return n, round(total / 1024 / 1024, 2)

zonas = {
    "Landing (eventos)": EVENTS_LANDING,
    "Bronze": BRONZE,
    "Silver": SILVER_EVENTS,
    "Gold": GOLD_USAGE,
    "Quarantine": QUARANTINE_EVENTS,
}

for nombre, camino in zonas.items():
    n, mb = tam(camino)
    print(f"{nombre:20s} {n:4d} archivos  {mb:8.2f} MB  {camino}")

Landing (eventos)     120 archivos     12.33 MB  /content/cloud-provider-analytics/data/extracted/datalake/landing/usage_events_stream
Bronze                209 archivos      3.38 MB  /content/cloud-provider-analytics/data/bronze
Silver                 40 archivos      2.47 MB  /content/cloud-provider-analytics/data/silver/usage_events
Gold                    2 archivos      0.27 MB  /content/cloud-provider-analytics/data/gold/org_daily_usage_by_service
Quarantine             20 archivos      0.13 MB  /content/cloud-provider-analytics/data/quarantine/usage_events


In [22]:
TEST_LATE = os.path.join(REPO, "data/test_late")
shutil.rmtree(TEST_LATE, ignore_errors=True)       # empezar limpio

late_stream = (
    spark.readStream
    .schema(schema_events)
    .option("maxFilesPerTrigger", 1)
    .json(EVENTS_LANDING)                          # archivos originales
    .withWatermark("timestamp", "10 minutes")      # watermark por fecha del evento
    .dropDuplicatesWithinWatermark(["event_id"])
)

q = (
    late_stream.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", os.path.join(TEST_LATE, "out"))
    .option("checkpointLocation", os.path.join(TEST_LATE, "checkpoint"))
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()

total = 43200
pasaron = spark.read.parquet(os.path.join(TEST_LATE, "out")).count()
descartados = total - pasaron
print("Con watermark por fecha del evento:")
print("  Pasaron el watermark:", pasaron)
print("  Descartados por llegar tarde:", descartados)
print("  % descartado:", round(descartados / total * 100, 2))

# Contraste: Bronze, con watermark por hora de llegada, no pierde nada
print("Bronze con watermark por ingest_ts:", spark.read.parquet(BRONZE_EVENTS).count())

# Idempotencia de Silver: re-ejecutar no cambia el conteo
antes = spark.read.parquet(SILVER_EVENTS).count()
run_silver_events()
despues = spark.read.parquet(SILVER_EVENTS).count()
print(f"Silver re-ejecutado -> antes={antes}  después={despues}")

inicio 71cf5b06-ffca-4b46-ac34-bb518d5c8fce None
fin 71cf5b06-ffca-4b46-ac34-bb518d5c8fce None
Con watermark por fecha del evento:
  Pasaron el watermark: 732
  Descartados por llegar tarde: 42468
  % descartado: 98.31
Bronze con watermark por ingest_ts: 43200
inicio 33522ed2-5356-4b13-8e5b-ecd7f0260d51 None
fin 33522ed2-5356-4b13-8e5b-ecd7f0260d51 None
Silver re-ejecutado -> antes=42989  después=42989


In [23]:
# Prueba controlada de late data (misma lógica que la Demo 4 de clase: "Evento muy tardío")
import json, shutil
from pathlib import Path
from datetime import datetime, timedelta

tardio_input = Path(REPO) / "data/test_tardio/in"
tardio_checkpoint = Path(REPO) / "data/test_tardio/checkpoint"

for p in [tardio_input, tardio_checkpoint]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def escribir_eventos(directorio, nombre, eventos):
    archivo = Path(directorio) / f"{nombre}.json"
    with archivo.open("w", encoding="utf-8") as f:
        for evento in eventos:
            fila = dict(evento)
            if isinstance(fila.get("timestamp"), datetime):
                fila["timestamp"] = fila["timestamp"].strftime("%Y-%m-%dT%H:%M:%S")
            f.write(json.dumps(fila, ensure_ascii=False) + "\n")
    print("Archivo creado:", archivo)

def evento(event_id, ts):
    return {"event_id": event_id, "timestamp": ts, "org_id": "org_test",
            "service": "compute", "metric": "requests", "value": "1", "unit": "count",
            "cost_usd_increment": 0.1, "schema_version": 2}

tardio_stream = (
    spark.readStream
    .schema(schema_events)
    .json(str(tardio_input))
    .withWatermark("timestamp", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)

for q in spark.streams.active:
    if q.name == "prueba_tardio":
        q.stop()

query_tardio = (
    tardio_stream.writeStream
    .format("memory")
    .queryName("prueba_tardio")
    .outputMode("append")
    .option("checkpointLocation", str(tardio_checkpoint))
    .trigger(processingTime="2 seconds")
    .start()
)

t0 = datetime(2025, 8, 1, 10, 0)

# Primer lote
escribir_eventos(tardio_input, "lote_001", [
    evento("t1", t0),
    evento("t2", t0 + timedelta(minutes=5)),
])
query_tardio.processAllAvailable()

# Segundo lote: hace avanzar el watermark a 10:20
escribir_eventos(tardio_input, "lote_002", [
    evento("t3", t0 + timedelta(minutes=30)),
])
query_tardio.processAllAvailable()

# Tercer lote: evento muy tardío (10:02, anterior al watermark)
escribir_eventos(tardio_input, "lote_003_tardio", [
    evento("t4", t0 + timedelta(minutes=2)),
])
query_tardio.processAllAvailable()

spark.sql("""
SELECT event_id, timestamp
FROM prueba_tardio
ORDER BY timestamp
""").show(truncate=False)

p = query_tardio.lastProgress
print("Event time / watermark:", p.get("eventTime", {}))
print("Descartados por watermark:", p["stateOperators"][0]["numRowsDroppedByWatermark"])
query_tardio.stop()

inicio 2b453c09-1549-4074-96ba-5a6faa6d34e7 prueba_tardio
Archivo creado: /content/cloud-provider-analytics/data/test_tardio/in/lote_001.json
Archivo creado: /content/cloud-provider-analytics/data/test_tardio/in/lote_002.json
Archivo creado: /content/cloud-provider-analytics/data/test_tardio/in/lote_003_tardio.json
+--------+-------------------+
|event_id|timestamp          |
+--------+-------------------+
|t1      |2025-08-01 10:00:00|
|t2      |2025-08-01 10:05:00|
|t3      |2025-08-01 10:30:00|
+--------+-------------------+

Event time / watermark: {'avg': '2025-08-01T10:02:00.000Z', 'max': '2025-08-01T10:02:00.000Z', 'min': '2025-08-01T10:02:00.000Z', 'watermark': '2025-08-01T10:20:00.000Z'}
Descartados por watermark: 1
fin 2b453c09-1549-4074-96ba-5a6faa6d34e7 None


In [24]:
# Prueba: con el watermark sobre ingest_ts (como en Bronze) el evento tardío no se descarta
ingest_input = Path(REPO) / "data/test_ingest/in"
ingest_checkpoint = Path(REPO) / "data/test_ingest/checkpoint"

for p in [ingest_input, ingest_checkpoint]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

ingest_stream = (
    spark.readStream
    .schema(schema_events)
    .json(str(ingest_input))
    .withColumn("ingest_ts", F.current_timestamp())
    .withWatermark("ingest_ts", "10 minutes")
    .dropDuplicatesWithinWatermark(["event_id"])
)

for q in spark.streams.active:
    if q.name == "prueba_ingest":
        q.stop()

query_ingest = (
    ingest_stream.writeStream
    .format("memory")
    .queryName("prueba_ingest")
    .outputMode("append")
    .option("checkpointLocation", str(ingest_checkpoint))
    .trigger(processingTime="2 seconds")
    .start()
)

# Los mismos tres lotes que en la prueba anterior
escribir_eventos(ingest_input, "lote_001", [
    evento("t1", t0),
    evento("t2", t0 + timedelta(minutes=5)),
])
query_ingest.processAllAvailable()

escribir_eventos(ingest_input, "lote_002", [
    evento("t3", t0 + timedelta(minutes=30)),
])
query_ingest.processAllAvailable()

escribir_eventos(ingest_input, "lote_003_tardio", [
    evento("t4", t0 + timedelta(minutes=2)),
])
query_ingest.processAllAvailable()

spark.sql("""
SELECT event_id, timestamp
FROM prueba_ingest
ORDER BY timestamp
""").show(truncate=False)

p = query_ingest.lastProgress
entraron = spark.sql("SELECT * FROM prueba_ingest").count()
descartados = p["stateOperators"][0]["numRowsDroppedByWatermark"]
print("Eventos que entraron:", entraron)
print("Descartados por watermark:", descartados)
query_ingest.stop()

if entraron == 4 and descartados == 0:
    print("OK: con el watermark sobre ingest_ts el evento tardío no se descarta")
else:
    print("FALLA: se descartó algún evento")

inicio c5991564-6265-4631-8292-bb1fff68ce26 prueba_ingest
Archivo creado: /content/cloud-provider-analytics/data/test_ingest/in/lote_001.json
Archivo creado: /content/cloud-provider-analytics/data/test_ingest/in/lote_002.json
Archivo creado: /content/cloud-provider-analytics/data/test_ingest/in/lote_003_tardio.json
+--------+-------------------+
|event_id|timestamp          |
+--------+-------------------+
|t1      |2025-08-01 10:00:00|
|t4      |2025-08-01 10:02:00|
|t2      |2025-08-01 10:05:00|
|t3      |2025-08-01 10:30:00|
+--------+-------------------+

Eventos que entraron: 4
Descartados por watermark: 0
fin c5991564-6265-4631-8292-bb1fff68ce26 None
OK: con el watermark sobre ingest_ts el evento tardío no se descarta


In [25]:
# Prueba: Gold saca los repetidos que llegaron tarde y Bronze no detectó
# (trabaja en memoria: no escribe en data/silver ni en data/gold)
silver = spark.read.parquet(SILVER_EVENTS)

# Agregamos a propósito los eventos de una organización que ya están en Silver
repetidos = silver.filter(F.col("org_id") == "org_xaji0y6d")
silver_con_repetidos = silver.union(repetidos)
print("Eventos en Silver:", silver.count())
print("Eventos repetidos agregados:", repetidos.count())
print("Silver con repetidos:", silver_con_repetidos.count())

gold_prueba = calcular_gold(silver_con_repetidos)
filas = gold_prueba.count()
eventos = gold_prueba.agg(F.sum("n_events")).first()[0]
print("Gold con repetidos -> filas:", filas)
print("Gold con repetidos -> suma n_events:", eventos)

if filas == 11050 and eventos == silver.count():
    print("OK: Gold sacó los repetidos y quedó igual que sin ellos")
else:
    print("FALLA: Gold contó eventos repetidos")

Eventos en Silver: 42989
Eventos repetidos agregados: 209
Silver con repetidos: 43198
Gold con repetidos -> filas: 11050
Gold con repetidos -> suma n_events: 42989
OK: Gold sacó los repetidos y quedó igual que sin ellos


In [26]:
# el keyspace lo creamos desde la página de Astra, porque Astra no deja crearlo con CQL

# prueba de conexión a Astra (Cassandra)
!pip -q install cassandra-driver

import glob
from google.colab import userdata
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

bundle = glob.glob("/content/secure-connect*.zip")[0]
# Removemos posibles comillas dobles que tenga el token guardado en Secrets
token = userdata.get("ASTRA_TOKEN").strip('"')

cluster = Cluster(
    cloud={"secure_connect_bundle": bundle},
    auth_provider=PlainTextAuthProvider("token", token),
)
session = cluster.connect(config["cassandra"]["keyspace"])

fila = session.execute("SELECT release_version FROM system.local").one()
print("Conectado. Versión de Cassandra:", fila.release_version)
print("Keyspace en uso:", session.keyspace)

Conectado. Versión de Cassandra: 4.0.11.0-6819aa9c7f6c
Keyspace en uso: cloud_provider


In [27]:
# tabla 1 en Cassandra + carga desde Gold
# Clave: partición por org_id y orden por (usage_date, service), porque las tres
# áreas consultan por organización y período (ver evidence/02_decisiones/decisiones.md).
from cassandra.concurrent import execute_concurrent_with_args

GOLD_PATH = ruta("gold_uso")

# si corremos esto dos veces no se duplican filas: Cassandra pisa la fila si ya existe la misma org, fecha y servicio
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_provider.org_daily_usage_by_service (
    org_id text,
    usage_date date,
    service text,
    daily_cost_usd double,
    requests bigint,
    cpu_hours double,
    storage_gb_hours double,
    genai_tokens bigint,
    carbon_kg double,
    n_events bigint,
    PRIMARY KEY ((org_id), usage_date, service)
) WITH CLUSTERING ORDER BY (usage_date ASC, service ASC)
""")

filas = spark.read.parquet(GOLD_PATH).collect()

insert = session.prepare("""
INSERT INTO cloud_provider.org_daily_usage_by_service
(org_id, usage_date, service, daily_cost_usd, requests, cpu_hours,
 storage_gb_hours, genai_tokens, carbon_kg, n_events)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

# requests viene como decimal pero son conteos, así que lo pasamos a entero. En la celda siguiente chequeamos que no se pierda nada
args = [(r["org_id"], r["usage_date"], r["service"], r["daily_cost_usd"],
         None if r["requests"] is None else int(r["requests"]),
         r["cpu_hours"], r["storage_gb_hours"], r["genai_tokens"],
         r["carbon_kg"], r["n_events"]) for r in filas]

resultados = execute_concurrent_with_args(session, insert, args, concurrency=50)
errores = [r for ok, r in resultados if not ok]

en_cassandra = session.execute(
    "SELECT COUNT(*) FROM cloud_provider.org_daily_usage_by_service").one()[0]
print("Filas en Gold:", len(filas))
print("Filas en Cassandra:", en_cassandra)
print("Errores de escritura:", len(errores))

Filas en Gold: 11050
Filas en Cassandra: 11050
Errores de escritura: 0


In [28]:
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_provider.org_top_services_14d (
    org_id text,
    window_end date,
    cost_usd_14d double,
    service text,
    PRIMARY KEY ((org_id, window_end), cost_usd_14d, service)
) WITH CLUSTERING ORDER BY (cost_usd_14d DESC, service ASC)
""")

filas_14d = spark.read.parquet(ruta("gold_servicios_14d")).collect()

insert_14d = session.prepare("""
INSERT INTO cloud_provider.org_top_services_14d (org_id, window_end, cost_usd_14d, service)
VALUES (?, ?, ?, ?)
""")
args_14d = [(r["org_id"], r["window_end"], r["cost_usd_14d"], r["service"]) for r in filas_14d]
resultados_14d = execute_concurrent_with_args(session, insert_14d, args_14d, concurrency=50)
errores_14d = [r for ok, r in resultados_14d if not ok]

en_cassandra_14d = session.execute("SELECT COUNT(*) FROM cloud_provider.org_top_services_14d").one()[0]
print("Filas en Gold:", len(filas_14d))
print("Filas en Cassandra:", en_cassandra_14d)
print("Errores de escritura:", len(errores_14d))

Filas en Gold: 262
Filas en Cassandra: 262
Errores de escritura: 0


In [29]:
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_provider.revenue_by_org_month (
    org_id text,
    month date,
    currency text,
    subtotal_usd double,
    credits_usd double,
    taxes_usd double,
    revenue_usd double,
    exchange_rate_to_usd double,
    n_facturas int,
    PRIMARY KEY ((org_id), month)
) WITH CLUSTERING ORDER BY (month ASC)
""")

filas_rev = spark.read.parquet(ruta("gold_revenue")).collect()

insert_rev = session.prepare("""
INSERT INTO cloud_provider.revenue_by_org_month
(org_id, month, currency, subtotal_usd, credits_usd, taxes_usd, revenue_usd, exchange_rate_to_usd, n_facturas)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""")
args_rev = [(r["org_id"], r["month"], r["currency"], r["subtotal_usd"], r["credits_usd"],
             r["taxes_usd"], r["revenue_usd"], r["exchange_rate_to_usd"], r["n_facturas"]) for r in filas_rev]
resultados_rev = execute_concurrent_with_args(session, insert_rev, args_rev, concurrency=50)
errores_rev = [r for ok, r in resultados_rev if not ok]

en_cassandra_rev = session.execute("SELECT COUNT(*) FROM cloud_provider.revenue_by_org_month").one()[0]
print("Filas en Gold:", len(filas_rev))
print("Filas en Cassandra:", en_cassandra_rev)
print("Errores de escritura:", len(errores_rev))

Filas en Gold: 240
Filas en Cassandra: 240
Errores de escritura: 0


In [30]:
from pyspark.sql import functions as F

g = spark.read.parquet(GOLD_PATH)
print("Gold: filas con requests no entero:",
      g.filter(F.col("requests") != F.col("requests").cast("long")).count())
print("Gold: suma de requests:", g.agg(F.sum("requests")).first()[0])

c = session.execute("SELECT SUM(requests) FROM cloud_provider.org_daily_usage_by_service").one()[0]
print("Cassandra: suma de requests:", c)

Gold: filas con requests no entero: 0
Gold: suma de requests: 2281020.0


Cassandra: suma de requests: 2281020


In [31]:
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_provider.genai_tokens_by_org_date (
    org_id text,
    usage_date date,
    genai_tokens bigint,
    cost_usd double,
    n_events bigint,
    PRIMARY KEY ((org_id), usage_date)
) WITH CLUSTERING ORDER BY (usage_date ASC)
""")

filas_genai = spark.read.parquet(ruta("gold_genai")).collect()

insert_genai = session.prepare("""
INSERT INTO cloud_provider.genai_tokens_by_org_date (org_id, usage_date, genai_tokens, cost_usd, n_events)
VALUES (?, ?, ?, ?, ?)
""")
args_genai = [(r["org_id"], r["usage_date"], r["genai_tokens"], r["cost_usd"], r["n_events"]) for r in filas_genai]
resultados_genai = execute_concurrent_with_args(session, insert_genai, args_genai, concurrency=50)
errores_genai = [r for ok, r in resultados_genai if not ok]

en_cassandra_genai = session.execute("SELECT COUNT(*) FROM cloud_provider.genai_tokens_by_org_date").one()[0]
print("Filas en Gold:", len(filas_genai))
print("Filas en Cassandra:", en_cassandra_genai)
print("Errores de escritura:", len(errores_genai))

Filas en Gold: 848
Filas en Cassandra: 848
Errores de escritura: 0


In [32]:
session.execute("""
CREATE TABLE IF NOT EXISTS cloud_provider.tickets_by_org_date (
    org_id text,
    severity text,
    ticket_date date,
    n_tickets bigint,
    n_sla_breached bigint,
    sla_breach_rate double,
    csat_avg double,
    n_open bigint,
    PRIMARY KEY ((org_id, severity), ticket_date)
) WITH CLUSTERING ORDER BY (ticket_date DESC)
""")

filas_tk = spark.read.parquet(ruta("gold_tickets")).collect()

insert_tk = session.prepare("""
INSERT INTO cloud_provider.tickets_by_org_date
(org_id, severity, ticket_date, n_tickets, n_sla_breached, sla_breach_rate, csat_avg, n_open)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")
args_tk = [(r["org_id"], r["severity"], r["ticket_date"], r["n_tickets"], r["n_sla_breached"],
            r["sla_breach_rate"], r["csat_avg"], r["n_open"]) for r in filas_tk]
resultados_tk = execute_concurrent_with_args(session, insert_tk, args_tk, concurrency=50)
errores_tk = [r for ok, r in resultados_tk if not ok]

en_cassandra_tk = session.execute("SELECT COUNT(*) FROM cloud_provider.tickets_by_org_date").one()[0]
print("Filas en Gold:", len(filas_tk))
print("Filas en Cassandra:", en_cassandra_tk)
print("Errores de escritura:", len(errores_tk))

Filas en Gold: 984
Filas en Cassandra: 984
Errores de escritura: 0


In [33]:
# consulta CQL 1 — costos y requests diarios de una organización en un rango
q1 = """
SELECT usage_date, service, daily_cost_usd, requests
FROM cloud_provider.org_daily_usage_by_service
WHERE org_id = 'org_dbdw2pcn'
  AND usage_date >= '2025-07-01'
  AND usage_date <= '2025-07-14'
"""
r1 = list(session.execute(q1))
for f in r1:
    print(f.usage_date, f.service, f.daily_cost_usd, f.requests)
print("Filas:", len(r1), "| costo total:", round(sum(f.daily_cost_usd for f in r1), 4))

2025-07-03 database 20.463 383
2025-07-04 database 9.1282 120
2025-07-05 compute 10.5084 110
2025-07-05 database 20.5993 367
2025-07-05 networking 1.806 125
2025-07-06 compute 26.692 299
2025-07-06 database 15.5938 328
2025-07-06 networking 0.3233 None
2025-07-07 compute 18.8965 267
2025-07-07 database 1.7029 None
2025-07-07 networking 2.6985 240
2025-07-08 compute 20.5009 251
2025-07-08 database 6.8366 110
2025-07-08 networking 1.3052 116
2025-07-09 compute 20.7514 257
2025-07-09 database 10.7993 232
2025-07-09 networking 1.5492 135
2025-07-10 compute 21.7875 277
2025-07-10 database 11.9865 216
2025-07-10 networking 0.2637 None
2025-07-11 compute 19.1478 226
2025-07-11 database 17.7013 357
2025-07-11 networking 0.9433 102
2025-07-12 compute 1.6501 None
2025-07-12 database 16.9153 352
2025-07-12 networking 1.6244 205
2025-07-13 compute 39.8767 459
2025-07-13 database 8.9409 134
2025-07-13 networking 0.0663 None
2025-07-14 compute 21.7431 245
2025-07-14 database 8.6132 146
Filas: 31 | c

In [34]:
# consulta CQL 2 — servicios que más costaron en los últimos 14 días de una organización
q2 = """
SELECT service, cost_usd_14d
FROM cloud_provider.org_top_services_14d
WHERE org_id = 'org_dbdw2pcn'
  AND window_end = '2025-08-31'
LIMIT 5
"""
r2 = list(session.execute(q2))
for f in r2:
    print(f.service, round(f.cost_usd_14d, 2))
print("Filas:", len(r2))

compute 278.1
database 133.29
networking 8.39
Filas: 3


In [37]:
# consulta CQL 3 — tickets críticos y tasa de SLA breach por día en los últimos 30 días de una organización
q3 = """
SELECT ticket_date, n_tickets, n_sla_breached, sla_breach_rate, csat_avg, n_open
FROM cloud_provider.tickets_by_org_date
WHERE org_id = 'org_w3zp08j3'
  AND severity = 'critical'
  AND ticket_date >= '2025-08-02'
  AND ticket_date <= '2025-08-31'
"""
r3 = list(session.execute(q3))
for f in r3:
    print(f.ticket_date, f.n_tickets, f.n_sla_breached, f.sla_breach_rate, f.csat_avg, f.n_open)
print("Filas:", len(r3), "| tickets críticos:", sum(f.n_tickets for f in r3))

2025-08-23 1 0 0.0 4.0 0
2025-08-08 1 0 0.0 3.0 1
Filas: 2 | tickets críticos: 2


In [35]:
# consulta CQL 4 — revenue mensual de una organización, con créditos e impuestos, en USD
q4 = """
SELECT month, currency, subtotal_usd, credits_usd, taxes_usd, exchange_rate_to_usd, revenue_usd
FROM cloud_provider.revenue_by_org_month
WHERE org_id = 'org_xaji0y6d'
"""
r4 = list(session.execute(q4))
for f in r4:
    print(f.month, f.currency, f.subtotal_usd, f.credits_usd, f.taxes_usd, f.exchange_rate_to_usd, f.revenue_usd)
print("Filas:", len(r4))

2025-06-01 USD 1107.09 0.0 232.49 0.97684 1339.58
2025-07-01 USD 621.22 0.0 130.46 1.00331 751.68
2025-08-01 ARS 0.87 0.0 0.18 0.0016 1.05
Filas: 3


In [36]:
# consulta CQL 5 — tokens GenAI y costo estimado por día de una organización
q5 = """
SELECT usage_date, genai_tokens, cost_usd, n_events
FROM cloud_provider.genai_tokens_by_org_date
WHERE org_id = 'org_pbhsahxt'
  AND usage_date >= '2025-08-18'
  AND usage_date <= '2025-08-31'
"""
r5 = list(session.execute(q5))
for f in r5:
    print(f.usage_date, f.genai_tokens, f.cost_usd, f.n_events)
print("Filas:", len(r5), "| tokens:", sum(f.genai_tokens for f in r5), "| costo total:", round(sum(f.cost_usd for f in r5), 4))

2025-08-18 4426 30.663 6
2025-08-19 2526 13.2515 3
2025-08-20 3255 58.5554 6
2025-08-21 8225 96.539 9
2025-08-22 4488 70.509 5
2025-08-23 2530 19.1928 3
2025-08-24 3190 19.3536 5
2025-08-25 8631 92.0452 9
2025-08-26 2940 23.4376 6
2025-08-27 1777 2.5819 3
2025-08-28 4885 43.9459 6
2025-08-29 2210 16.7402 3
2025-08-30 5401 34.0184 6
2025-08-31 2488 24.7227 3
Filas: 14 | tokens: 56972 | costo total: 545.5562


In [38]:
# Pruebas automáticas: se corren con un solo comando, sin ejecutar celdas
!cd {REPO} && python tests/test_pipeline.py

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 18:41:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where